# 03 — AGT-B Behavior

Runtime safety constraints that run *after* AGT-A passes.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.agents.inventory import InventoryAgent
from src.agents.order_processing import OrderProcessingAgent
from src.agents.returns import ReturnsAgent
from src.models import AgentRole

## Rate Limiting

In [ ]:
inv = InventoryAgent(role=AgentRole.WAREHOUSE_MANAGER)
# Simulate 101 recent actions (AGT-B: max 100)
inv._recent_actions = [{'action': 'read'}] * 101
result = inv.invoke_tool('check_stock', {'product_id': 'SKU-100', 'location': 'STORE-NYC'})
print('Rate limited:', result)

## Large Inventory Transfers

In [ ]:
# Warehouse staff blocked for transfers > 500 units (AGT-B)
staff = InventoryAgent(role=AgentRole.WAREHOUSE_STAFF)
result = staff.invoke_tool('transfer_inventory', {'product_id': 'SKU-100', 'from_location': 'WAREHOUSE-EAST', 'to_location': 'STORE-NYC', 'quantity': 600})
print('Staff 600 units:', result)

# Manager allowed
mgr = InventoryAgent(role=AgentRole.WAREHOUSE_MANAGER)
result = mgr.invoke_tool('transfer_inventory', {'product_id': 'SKU-100', 'from_location': 'WAREHOUSE-EAST', 'to_location': 'STORE-NYC', 'quantity': 600})
print('Manager 600 units:', result)

## Pricing Rules

In [ ]:
op = OrderProcessingAgent(role=AgentRole.STORE_MANAGER)

# Below cost: SKU-300 (cost=$25), 75% off -> $22.50 < $25
result = op.invoke_tool('apply_discount', {'product_id': 'SKU-300', 'discount_percent': 75.0})
print('Below cost:', result)

# Below MAP: SKU-100 (MAP=$149.99), 30% off -> $139.99 < $149.99
result = op.invoke_tool('apply_discount', {'product_id': 'SKU-100', 'discount_percent': 30.0})
print('Below MAP:', result)

# Valid: 10% off SKU-300
result = op.invoke_tool('apply_discount', {'product_id': 'SKU-300', 'discount_percent': 10.0})
print('Valid discount:', result)

## Refund Rules

In [ ]:
ret = ReturnsAgent(role=AgentRole.STORE_MANAGER)

# ORD-002: 45 days since purchase -> blocked
result = ret.invoke_tool('process_refund', {'order_id': 'ORD-002', 'amount': 89.99})
print('Late refund:', result)

# ORD-001: 5 days, has receipt -> allowed
result = ret.invoke_tool('process_refund', {'order_id': 'ORD-001', 'amount': 30.0})
print('Valid refund:', result)